# Norton–Baker: combine structure + activity (one row per variant)

**Kernel:** select `PETase structure-activity` (conda env `petase_structure_activity`).

**Problem with the old notebook:** `pd.merge(df_rounds, df_activity, on='sample_id')` creates a *cartesian product* — every activity measurement is crossed with every Rounds row for that enzyme (3,600+ rows instead of 214).

**Fix:**
1. Collapse replicate assay wells → **one activity value per variant**
2. Keep **one sequence/metadata row per variant** from Rounds
3. Merge on `sample_id` (= `base_name` in the structure file)

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('.')
XLSX = DATA_DIR / 'Norton_Baker_supplementary.xlsx'
STRUCTURE_CSV = DATA_DIR / 'nb_petase_final_summary.csv'
OUT_CSV = DATA_DIR / 'NB_PETase_with_activity.csv'

## Step 1 — Load raw tables

In [ ]:
df_rounds = pd.read_excel(XLSX, sheet_name='Rounds')
df_activity = pd.read_excel(XLSX, sheet_name='Activity')
df_structure = pd.read_csv(STRUCTURE_CSV)

print('Rounds rows:', len(df_rounds), '| unique enzymes:', df_rounds['sample_id'].nunique())
print('Activity rows:', len(df_activity), '| unique enzymes:', df_activity['sample_id'].nunique())
print('Structure rows:', len(df_structure))

## Step 2 — Pick the assay condition for each variant

Norton–Baker ran enzymes in three screening rounds (see paper Table S2):
- `DP*` → **P740 Round1** (pH 5.5 only — Round 1 did not test pH 4.5)
- `ESM*` → **P740 ESM** (pH 4.5)
- `TEP*` → **P740 TEP** (pH 4.5)
- controls (`LCC-ICCG`, `RFP`) → any dataset at pH 4.5

We use **crystalline powder (cryPow) at 40 °C**, the paper's main industrially relevant substrate.

In [ ]:
SUBSTRATE = 'cryPow'
TEMPERATURE_C = 40

def activity_dataset(sample_id: str):
    if sample_id.startswith('DP'):
        return 'P740 Round1'
    if sample_id.startswith('ESM'):
        return 'P740 ESM'
    if sample_id.startswith('TEP'):
        return 'P740 TEP'
    return None  # controls

def activity_pH(sample_id: str):
    return 5.5 if sample_id.startswith('DP') else 4.5

## Step 3 — Collapse replicates with the **median**

**Why median (not mean)?**
- The paper ran **biological duplicates** (wells on plates `1A` and `1B`) for high-yield enzymes.
- Some plates (`3A`, `3B`, …) are empty controls and read as 0 — median is robust to those outliers.
- With only 2 replicates, median = mean; with 1 measurement, median = that value.

We prefer `1A`/`1B` wells when present; otherwise we use all wells at that condition.

In [ ]:
df_activity['plate_prefix'] = df_activity['well'].astype(str).str.split('-').str[0]

ACTIVITY_COL = 'percent_depolymerization'
ACTIVITY_COL_2 = 'umol_product_per_mg_enzyme'  # paper's primary activity unit

collapsed_rows = []
skip_ids = {'blank', 'sumo_LCC-ICCG'}

for sample_id in df_activity['sample_id'].dropna().unique():
    if sample_id in skip_ids:
        continue

    dataset = activity_dataset(sample_id)
    pH = activity_pH(sample_id)

    mask = (
        (df_activity['sample_id'] == sample_id)
        & (df_activity['substrate'] == SUBSTRATE)
        & (df_activity['pH'] == pH)
        & (df_activity['temperature_c'] == TEMPERATURE_C)
    )
    if dataset is not None:
        mask &= df_activity['dataset'] == dataset

    reps = df_activity.loc[mask]
    if reps.empty:
        continue

    # biological duplicate plates when available
    ab = reps[reps['plate_prefix'].isin(['1A', '1B'])]
    reps_use = ab if not ab.empty else reps

    collapsed_rows.append({
        'sample_id': sample_id,
        'activity_dataset': dataset if dataset else reps.iloc[0]['dataset'],
        'activity_pH': pH,
        'activity_temperature_c': TEMPERATURE_C,
        'activity_substrate': SUBSTRATE,
        'n_replicates': len(reps_use),
        'percent_depolymerization_median': reps_use[ACTIVITY_COL].astype(float).median(),
        'percent_depolymerization_mean': reps_use[ACTIVITY_COL].astype(float).mean(),
        'umol_product_per_mg_enzyme_median': reps_use[ACTIVITY_COL_2].astype(float).median(),
    })

df_activity_collapsed = pd.DataFrame(collapsed_rows)
print('Collapsed activity table:', len(df_activity_collapsed), 'variants')
print('Replicates used per variant:')
print(df_activity_collapsed['n_replicates'].value_counts().sort_index())
df_activity_collapsed.head()

## Step 4 — One metadata row per variant from Rounds

In [ ]:
# Rounds has multiple rows per enzyme (different plates/rounds) — keep one row each
df_rounds_one = (
    df_rounds.sort_values('round')
    .drop_duplicates('sample_id', keep='first')
    .reset_index(drop=True)
)
print('Rounds deduplicated:', len(df_rounds_one), 'variants')

## Step 5 — Merge: sequence + collapsed activity + structure features

Join key: `sample_id` (activity/rounds) = `base_name` (structure CSV)

In [ ]:
df_merged = (
    df_rounds_one
    .merge(df_activity_collapsed, on='sample_id', how='inner')
    .merge(
        df_structure[['base_name', 'entity_1_sequence', 'confidence_score_model_0', 'complex_plddt_model_0']],
        left_on='sample_id',
        right_on='base_name',
        how='left',
    )
)

print('Final table:', len(df_merged), 'rows (should be 214)')
print('Missing activity:', df_merged['percent_depolymerization_median'].isna().sum())
print('Missing structure:', df_merged['entity_1_sequence'].isna().sum())
df_merged[['sample_id', 'n_replicates', 'percent_depolymerization_median', 'umol_product_per_mg_enzyme_median', 'complex_plddt_model_0']].head(10)

## Step 6 — Save

In [ ]:
df_merged.to_csv(OUT_CSV, index=False)
print('Saved →', OUT_CSV.resolve())

## Sanity check — why the old merge was wrong

In [ ]:
bad_merge = df_rounds.merge(df_activity, on='sample_id', how='inner')
print('OLD merge (no collapsing):', len(bad_merge), 'rows')
print('NEW merge (correct):', len(df_merged), 'rows')